In [ ]:
# ==========================================
# Célula 1 - Configurações e Setup (somente extração)
# ==========================================
# Este notebook SÓ baixa os relatórios brutos do MKTZap (Analítico, Enquetes e Tickets dos 3 tenants).
# Não consolida, não trata e não grava em nenhuma base: os arquivos ficam na pasta local PASTA_SAIDA.
import os
import time
import glob
import json
import shutil
import stat
import subprocess
from datetime import datetime, timedelta

# Selenium e Webdriver
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Pasta no Drive dos arquivos Gold (pre-processing). Usada aqui também para saber se o mês anterior já foi fechado.
RAIZ_AUTO = r"G:\Drives compartilhados\REPÓSITORIOS - DOCUMENTAÇÕES\Dashboards - Tableau\Central de EXP e Serviços\Fontes de Dados\auto\mktzap"
# Os consolidados são gravados direto nas pastas oficiais de Fontes de Dados, lidas pelos fluxos e pelo pipeline.
RAIZ_FONTES = os.path.dirname(os.path.dirname(RAIZ_AUTO))

# Mês de referência:
#   None           = automático: sempre o mês atual (do dia 1º até agora) e, se o mês anterior ainda não foi fechado
#                    (virada num fim de semana ou feriado prolongado, execução que falhou...), também o mês anterior INTEIRO.
#                    Essa conferência só é feita até o dia JANELA_FECHAMENTO_DIAS do mês.
#   "mes_atual"    = só o mês atual.
#   "mes_anterior" = só o mês passado inteiro.
#   "ambos"        = mês passado inteiro + mês atual, sem conferir nada.
MES_REF = None
JANELA_FECHAMENTO_DIAS = 10

# Bases que têm Gold (o Tickets do NCC não é tratado). Um mês está FECHADO quando todas têm um Gold do mês inteiro gerado
# DEPOIS do último dia dele (o nome do Gold começa pela data da execução), ou seja, quando os últimos dias já entraram.
GOLDS_DO_MES = [
    ("27 - Brasil", "brasil", "analitico"), ("27 - Brasil", "brasil", "enquetes"), ("27 - Brasil", "brasil", "tickets"),
    ("1531 - Hispana", "hispana", "analitico"), ("1531 - Hispana", "hispana", "enquetes"), ("1531 - Hispana", "hispana", "tickets"),
    ("1990 - NCC", "ncc", "analitico"), ("1990 - NCC", "ncc", "enquetes"),
]


def montar_periodo(fim, fechamento):
    """Do dia 1º do mês de `fim` até `fim`. fechamento=True: mês passado inteiro (fim = 23:59:59 do último dia)."""
    ini = fim.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    return {
        "ini": ini,
        "fim": fim,
        "fechamento": fechamento,
        "pasta_mes": ini.strftime("%Y%m"),
        "atalho": "Mês passado" if fechamento else "Este mês",   # atalho de data do relatório Analítico
        "de_str": ini.strftime("%d/%m/%Y"),    # campos de data dos relatórios antigos (Tickets)
        "ate_str": fim.strftime("%d/%m/%Y"),
        "de_iso": ini.strftime("%Y-%m-%d"),    # campos <input type="date"> (Enquetes)
        "ate_iso": fim.strftime("%Y-%m-%d"),
        "de_keys": ini.strftime("%d%m%Y"),     # digitação nativa via teclado
        "ate_keys": fim.strftime("%d%m%Y"),
    }


def golds_pendentes(p):
    """Bases sem Gold do mês inteiro de `p` gerado depois do fim do mês (lista vazia = mês fechado)."""
    pendentes = []
    for nome_tenant, slug, relatorio in GOLDS_DO_MES:
        padrao = os.path.join(RAIZ_AUTO, "pre-processing", p["pasta_mes"], nome_tenant,
                              f"*-{slug}-{relatorio}-periodo-{p['de_iso']}-a-{p['ate_iso']}.csv")
        if not any(os.path.basename(a)[:10] > p["ate_iso"] for a in glob.glob(padrao)):
            pendentes.append(f"{slug}-{relatorio}")
    return pendentes


agora = datetime.now()
mes_atual = montar_periodo(agora, fechamento=False)
mes_anterior = montar_periodo(mes_atual["ini"] - timedelta(seconds=1), fechamento=True)

if MES_REF == "mes_atual":
    PERIODOS = [mes_atual]
elif MES_REF == "mes_anterior":
    PERIODOS = [mes_anterior]
elif MES_REF == "ambos":
    PERIODOS = [mes_anterior, mes_atual]
elif MES_REF is None:
    pendentes = golds_pendentes(mes_anterior)
    if pendentes and agora.day <= JANELA_FECHAMENTO_DIAS:
        PERIODOS = [mes_anterior, mes_atual]
        print(f"🔁 {mes_anterior['ini']:%m/%Y} ainda não foi fechado (sem Gold do mês inteiro gerado depois de "
              f"{mes_anterior['ate_str']}: {', '.join(pendentes)}). Ele será baixado de novo inteiro, junto com o mês atual.")
    else:
        PERIODOS = [mes_atual]
        if pendentes:
            print(f"⚠️ {mes_anterior['ini']:%m/%Y} não consta como fechado ({', '.join(pendentes)}), mas já passou do dia "
                  f"{JANELA_FECHAMENTO_DIAS}. Se precisar, rode com MES_REF = \"ambos\" ou \"mes_anterior\".")
else:
    raise ValueError(f"MES_REF inválido: {MES_REF!r}. Use None, 'mes_atual', 'mes_anterior' ou 'ambos'.")

# URLs do MKTZap
URL_MKTZAP = "https://natura.mktzap.com.br"
URL_LOGIN = f"{URL_MKTZAP}/login"
URL_CHAT = f"{URL_MKTZAP}/chat"
URL_ANALITICO = f"{URL_MKTZAP}/reports/analytic"
URL_ENQUETES = f"{URL_MKTZAP}/new/reports/polls"
URL_TICKETS = f"{URL_MKTZAP}/reports/history-tickets"
URL_BACKGROUND = f"{URL_MKTZAP}/new/reports/background-reports"

caminho_base = os.getcwd()   # pasta local: perfil do Chrome e downloads temporários

# Onde os relatórios brutos são guardados: <PASTA_SAIDA>\<anomês do período>\<tenant>\bruto_<relatorio>_<de>_a_<ate>.csv
PASTA_SAIDA = os.path.join(caminho_base, "downloads_brutos")

# Mapeamento Multi-Tenant
def montar_tenant(id_tenant, nome_exibicao, nome_empresa_mktzap):
    return {
        "id": id_tenant,
        "nome_exibicao": nome_exibicao,
        "nome_empresa_mktzap": nome_empresa_mktzap,
        "pasta_temp": os.path.join(caminho_base, f"downloads_temp_{id_tenant}"),
    }

def pasta_bruto(t, p):
    """Pasta dos brutos deste tenant no mês do período `p`."""
    return os.path.join(PASTA_SAIDA, p["pasta_mes"], t["nome_exibicao"])


tenants = [
    montar_tenant("27", "27 - Brasil", "27 - Natura - CN"),
    montar_tenant("1531", "1531 - Hispana", "1531 - Natura - Experiencia CN OIs"),
    montar_tenant("1990", "1990 - NCC", "1990 - Natura - NCC"),
]

# Garantia da existência das pastas físicas
for t in tenants:
    os.makedirs(t["pasta_temp"], exist_ok=True)
    for p in PERIODOS:
        os.makedirs(pasta_bruto(t, p), exist_ok=True)

# Estrutura Global para Rastreamento de Erros de Extração
registro_execucao = []

print("✅ Setup concluído! Período(s):")
for p in PERIODOS:
    print(f"   - {p['de_str']} até {p['ate_str']}" + ("  (mês anterior inteiro, fechamento)" if p["fechamento"] else ""))
print(f"   Saída dos brutos: {PASTA_SAIDA}")

In [ ]:
# ==========================================
# Célula 2 - Inicialização do Chrome e Login com Persistência
# ==========================================
# Pasta local para salvar o perfil do Chrome e manter o login ativo nas próximas execuções.
# Perfil só do MKTZap (login manual na 1ª vez): não faça login na conta Google nele (vira perfil gerenciado
# pela empresa, que abre abas de início e sincronização e atrapalha o Selenium).
pasta_perfil_chrome = os.path.join(caminho_base, "chrome_profile_mktzap")


def fechar_chrome_do_pipeline(perfil=pasta_perfil_chrome):
    """Fecha sessões anteriores que ficaram abertas ou travadas: todo chromedriver.exe e todo Chrome
    usando o perfil do pipeline. O Chrome do dia a dia (outro perfil) não é tocado."""
    alvo = os.path.normcase(os.path.join(caminho_base, "chrome_profile"))   # prefixo: pega todos os perfis do pipeline
    consulta = ("Get-CimInstance Win32_Process -Filter \"Name='chrome.exe' or Name='chromedriver.exe'\" | "
                "Select-Object ProcessId,Name,CommandLine | ConvertTo-Json -Compress")
    try:
        saida = subprocess.run(["powershell", "-NoProfile", "-Command", consulta],
                               capture_output=True, text=True, timeout=60).stdout.strip()
        processos = json.loads(saida) if saida else []
    except Exception as e:
        print(f"⚠️ Não foi possível listar os processos do Chrome: {e}")
        return
    if isinstance(processos, dict):
        processos = [processos]
    drivers = [p for p in processos if p["Name"].lower() == "chromedriver.exe"]
    chromes = [p for p in processos if p["Name"].lower() == "chrome.exe" and alvo in os.path.normcase(p.get("CommandLine") or "")]
    principais = [p for p in chromes if "--type=" not in (p.get("CommandLine") or "")] or chromes
    for p in drivers + principais:
        subprocess.run(["taskkill", "/F", "/T", "/PID", str(p["ProcessId"])], capture_output=True)
    if drivers or chromes:
        print(f"🧹 Sessões anteriores encerradas: {len(drivers)} chromedriver(s) e {len(principais)} Chrome(s) do pipeline.")
        time.sleep(2)
    for trava in ("lockfile", "SingletonLock", "SingletonCookie", "SingletonSocket"):   # travas de um Chrome morto à força
        try:
            os.remove(os.path.join(perfil, trava))
        except OSError:
            pass


# Se a célula for rodada de novo no mesmo kernel, fecha o navegador anterior antes
if "driver" in globals():
    try:
        driver.quit()
    except Exception:
        pass
fechar_chrome_do_pipeline()

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-popup-blocking")
chrome_options.add_argument("--disable-features=DownloadBubble,DownloadBubbleV2")
chrome_options.add_argument("--no-first-run")
chrome_options.add_argument("--no-default-browser-check")
chrome_options.add_argument(f"--user-data-dir={pasta_perfil_chrome}")

prefs = {
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
    "profile.default_content_settings.popups": 0
}
chrome_options.add_experimental_option("prefs", prefs)

print("🚀 Iniciando navegador do Selenium com perfil persistente...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

# Abas extras abertas na inicialização (restauração/política) chegam um instante depois: espera e fecha todas menos uma
time.sleep(3)
abas = driver.window_handles
for aba in abas[1:]:
    driver.switch_to.window(aba)
    driver.close()
driver.switch_to.window(abas[0])

wait_longo = WebDriverWait(driver, 600)  # Aguarda login manual se necessário
wait_curto = WebDriverWait(driver, 15)   # Esperas explícitas rápidas

driver.get(URL_LOGIN)
print("⏳ Verificando sessão ativa ou aguardando login no MKTZap...")

try:
    wait_longo.until(EC.url_contains("/chat"))
    print("✅ Login confirmado e sessão pronta para uso!")
except Exception as e:
    print(f"⚠️ Atenção ao verificar login: {e}")

In [ ]:
# ==========================================
# Célula 3 - Funções Auxiliares (CDP, Troca de Empresa e Downloads)
# ==========================================
def definir_pasta_download(pasta_temp):
    driver.execute_cdp_cmd("Page.setDownloadBehavior", {
        "behavior": "allow",
        "downloadPath": pasta_temp
    })

def trocar_empresa(nome_empresa):
    print(f"🏢 Alternando para a empresa: '{nome_empresa}'...")

    # Redireciona para o /chat antes de trocar a empresa para garantir menu lateral visível
    if "/chat" not in driver.current_url:
        driver.get(URL_CHAT)
        wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-bottom-a-dropdown']")))

    menu_user = wait_curto.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-test='menu-user-bottom-a-dropdown']")))
    driver.execute_script("arguments[0].scrollIntoView(true);", menu_user)
    driver.execute_script("arguments[0].click();", menu_user)

    btn_alterar = wait_curto.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-test='menu-user-bottom-a-change-company']")))
    driver.execute_script("arguments[0].click();", btn_alterar)

    input_empresa = wait_curto.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-bottom-modal-change-company-text-name']")))
    input_empresa.clear()

    for letra in nome_empresa:
        input_empresa.send_keys(letra)
        time.sleep(0.02)

    item_lista = wait_curto.until(EC.element_to_be_clickable((By.XPATH, f"//li[contains(text(), '{nome_empresa}')]")))
    driver.execute_script("arguments[0].click();", item_lista)

    btn_trocar = wait_curto.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-test='menu-user-botton-modal-change-company-button-ok']")))
    driver.execute_script("arguments[0].click();", btn_trocar)

    # Espera explícita para o recarregamento do painel
    time.sleep(4)

def aguardar_e_mover_download(pasta_temp, pasta_destino, prefixo_arquivo, p):
    inicio_espera = time.time()
    arquivo_baixado = None

    while time.time() - inicio_espera < 180: # Limite de 3 minutos
        # Só arquivos de relatório: ignora pacotes de extensão (.crx) e outros downloads estranhos do Chrome
        arquivos = [f for f in glob.glob(os.path.join(pasta_temp, "*.*")) if f.lower().endswith(('.csv', '.xlsx', '.xls', '.txt'))]
        if arquivos:
            arquivo_baixado = max(arquivos, key=os.path.getmtime)
            if os.path.getmtime(arquivo_baixado) >= inicio_espera - 10:
                # O Chrome pode criar o arquivo antes de terminar de gravar: espera ter conteúdo, tamanho estável e nenhum .crdownload
                estavel, ultimo_tamanho = 0, -1
                while time.time() - inicio_espera < 180:
                    em_andamento = glob.glob(os.path.join(pasta_temp, "*.crdownload"))
                    tamanho = os.path.getsize(arquivo_baixado) if os.path.exists(arquivo_baixado) else 0
                    estavel = estavel + 1 if (tamanho > 0 and tamanho == ultimo_tamanho and not em_andamento) else 0
                    if estavel >= 2:
                        break
                    ultimo_tamanho = tamanho
                    time.sleep(1.5)
                break
        time.sleep(2)

    if arquivo_baixado:
        ext = arquivo_baixado.split('.')[-1]
        nome_final = f"{prefixo_arquivo}_{p['ini']:%Y%m%d}_a_{p['fim']:%Y%m%d}.{ext}"
        caminho_final = os.path.join(pasta_destino, nome_final)

        if os.path.exists(caminho_final):
            os.remove(caminho_final)

        shutil.move(arquivo_baixado, caminho_final)
        print(f"  -> ✅ Arquivo salvo em: {nome_final}")
        return True
    else:
        print(f"  -> ⚠️ Timeout: O arquivo do relatório não apareceu na pasta de downloads.")
        return False

def baixar_do_background(t, p, prefixo_arquivo, nome_relatorio, limite_seg=300):
    """Abre a fila de Relatórios em Background, aguarda 'Finalizado' na 1ª linha e baixa o arquivo."""
    print("  -> Redirecionando para a fila de Relatórios em Background...")
    driver.get(URL_BACKGROUND)
    print("  -> Aguardando processamento da plataforma (Status: Finalizado)...")

    inicio_espera = time.time()
    while time.time() - inicio_espera < limite_seg:
        try:
            primeira_linha = wait_curto.until(EC.presence_of_element_located((By.XPATH, "//table//tbody//tr[1]")))
            if "Finalizado" in primeira_linha.text:
                print("  -> ✅ Status 'Finalizado' detectado! Iniciando download...")
                btn_dl = primeira_linha.find_element(By.XPATH, ".//button[@title='Download' or contains(@class, 'btn-outline-primary')]")
                driver.execute_script("arguments[0].click();", btn_dl)
                return aguardar_e_mover_download(t["pasta_temp"], pasta_bruto(t, p), prefixo_arquivo, p)
        except Exception:
            pass
        time.sleep(4)
        driver.refresh()

    print(f"  -> ⚠️ Timeout atingido ({limite_seg // 60} min) aguardando finalização no background.")
    registro_execucao.append({"tenant": t["nome_exibicao"], "periodo": p["pasta_mes"], "relatorio": nome_relatorio, "status": "Erro / Timeout Background"})
    return False

print("✅ Funções auxiliares carregadas com esperas explícitas!")

In [ ]:
# ==========================================
# Célula 4 - Extração do Relatório Analítico (função por tenant)
# ==========================================
def extrair_analitico(t, p):
    """Baixa o relatório deste tenant. A empresa já foi selecionada pela Célula 7 (troca feita uma única vez por tenant)."""
    print(f"\n" + "="*50)
    print(f"🚀 INICIANDO EXTRAÇÃO ANALÍTICO: {t['nome_exibicao']}")
    print(f"   -> Empresa MKTZap: {t['nome_empresa_mktzap']}")
    print(f"   -> Período:        {p['de_str']} até {p['ate_str']}")
    print(f"   -> Destino Final:  {pasta_bruto(t, p)}")
    print("="*50)

    try:

        driver.get(URL_ANALITICO)

        # 1. Clica no atalho de data ("Este mês" = mês corrente até agora; "Mês passado" = o mês passado inteiro)
        print(f"  -> Clicando no atalho de data: '{p['atalho']}'...")
        opcao_data = wait_curto.until(
            EC.element_to_be_clickable((By.XPATH, f"//a[contains(text(), '{p['atalho']}')]"))
        )
        driver.execute_script("arguments[0].click();", opcao_data)

        # 2. Garante seleção do Formato CSV
        try:
            select_export = Select(wait_curto.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "select[ng-model='vm.exportAs']"))
            ))
            select_export.select_by_value("csv")
            print("  -> Formato CSV selecionado com sucesso.")
        except Exception as e_format:
            print(f"  -> Aviso ao selecionar formato CSV: {e_format}")

        # 3. Clica em "Gerar Relatório"
        print("  -> Clicando em 'Gerar Relatório' e enviando para background...")
        btn_gerar = wait_curto.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button[ga-event='mktzap_bt_action_analytic_report_generate']"))
        )
        driver.execute_script("arguments[0].click();", btn_gerar)

        # Confirma modal de segundo plano, se exibido
        time.sleep(1.5)
        modais = driver.find_elements(By.XPATH, "//button[contains(text(),'Confirmar')]")
        if modais and modais[0].is_displayed():
            driver.execute_script("arguments[0].click();", modais[0])

        # 4 e 5. Fila de background -> "Finalizado" -> download
        time.sleep(5)   # dá tempo ao servidor de registrar o pedido (evita ler o relatório anterior da fila)
        baixar_do_background(t, p, "bruto_analitico", "Analítico")

    except Exception as e:
        print(f"❌ Erro crítico na extração do Analítico ({t['nome_exibicao']}): {e}")
        registro_execucao.append({"tenant": t["nome_exibicao"], "periodo": p["pasta_mes"], "relatorio": "Analítico", "status": f"Erro: {str(e)}"})

In [ ]:
# ==========================================
# Célula 5 - Extração do Relatório de Enquetes (função por tenant)
# ==========================================
# Enquete alvo por tenant (comparação em minúsculas, por trecho do nome)
ENQUETE_ALVO = {
    "27": "c-sat (experiência cnf)",
    "1531": "c-sat - satisfaccion",
    "1990": "c-sat - ncc (new)",
}

def extrair_enquetes(t, p):
    """Baixa o relatório deste tenant. A empresa já foi selecionada pela Célula 7 (troca feita uma única vez por tenant)."""
    print(f"\n==========================================")
    print(f"📊 Processando Enquetes: {t['nome_exibicao']} ({p['de_str']} até {p['ate_str']})")
    print(f"==========================================")

    try:

        driver.get(URL_ENQUETES)
        time.sleep(2)

        # 1. Localiza os campos de data
        input_from = wait_curto.until(EC.presence_of_element_located((By.ID, "polls-from")))
        input_to = driver.find_element(By.ID, "polls-to")

        print(f"  -> Injetando período ({p['de_iso']} até {p['ate_iso']})...")

        # Camada A: Digitação nativa via Keys
        try:
            input_from.click()
            input_from.send_keys(Keys.CONTROL + "a")
            input_from.send_keys(p["de_keys"])

            input_to.click()
            input_to.send_keys(Keys.CONTROL + "a")
            input_to.send_keys(p["ate_keys"])
        except Exception:
            pass

        # Camada B e C: Injeção no DOM + Controlador ngModel do Angular
        driver.execute_script("""
            var elFrom = arguments[0];
            var elTo = arguments[1];
            var valFrom = arguments[2];
            var valTo = arguments[3];

            function forcarInputAngular(el, val) {
                el.value = val;
                el.setAttribute('value', val);

                if (typeof angular !== 'undefined') {
                    var $el = angular.element(el);
                    var ngModel = $el.controller('ngModel');
                    if (ngModel) {
                        ngModel.$setViewValue(val);
                        ngModel.$render();
                    }
                    $el.triggerHandler('input');
                    $el.triggerHandler('change');
                }
            }

            forcarInputAngular(elFrom, valFrom);
            forcarInputAngular(elTo, valTo);
        """, input_from, input_to, p["de_iso"], p["ate_iso"])

        time.sleep(1)

        val_f = input_from.get_attribute("value")
        val_t = input_to.get_attribute("value")
        print(f"  -> Confirmado nos campos: De [{val_f}] Até [{val_t}]")

        # 2. Aguarda carregamento das opções no dropdown #polls-poll
        print("  -> Aguardando carregamento das enquetes no dropdown...")
        wait_curto.until(lambda d: len(Select(d.find_element(By.ID, "polls-poll")).options) > 1)

        dropdown_poll = Select(driver.find_element(By.ID, "polls-poll"))

        # 3. Mapeamento Exato da Enquete por Tenant
        texto_alvo = ENQUETE_ALVO.get(t["id"], "c-sat")

        encontrado = False
        for opt in dropdown_poll.options:
            if texto_alvo in opt.text.lower():
                dropdown_poll.select_by_visible_text(opt.text)
                print(f"  -> Enquete '{opt.text}' selecionada com sucesso.")
                encontrado = True
                break

        if not encontrado:
            dropdown_poll.select_by_index(1)
            print(f"  -> ⚠️ Alvo não encontrado. Selecionada a primeira disponível: '{dropdown_poll.first_selected_option.text}'")

        time.sleep(1.5)

        # 4. Clica em "Baixar detalhes da enquete"
        print("  -> Clicando em 'Baixar detalhes da enquete'...")
        btn_baixar = wait_curto.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Baixar detalhes da enquete')]")))
        driver.execute_script("arguments[0].click();", btn_baixar)

        # 5. Tratamento de Envio para Background (Modal de Confirmação)
        time.sleep(2)
        modais = driver.find_elements(By.XPATH, "//button[contains(@class, 'btn-primary') and contains(., 'Confirmar')]")

        if modais and modais[0].is_displayed():
            print("  -> Modal detetado. Enviando para fila de Background...")
            driver.execute_script("arguments[0].click();", modais[0])

            print("  -> Aguardando 5 segundos para o servidor registar o pedido...")
            time.sleep(5) # Evita a Race Condition de ler o relatório do Tenant anterior

            baixar_do_background(t, p, "bruto_enquetes", "Enquetes")
        else:
            print("  -> Download direto detetado. A guardar ficheiro...")
            aguardar_e_mover_download(t["pasta_temp"], pasta_bruto(t, p), "bruto_enquetes", p)

    except Exception as e:
        print(f"❌ Erro na extração de Enquetes ({t['nome_exibicao']}): {e}")
        registro_execucao.append({"tenant": t["nome_exibicao"], "periodo": p["pasta_mes"], "relatorio": "Enquetes", "status": f"Erro: {str(e)}"})

In [ ]:
# ==========================================
# Célula 6 - Extração do Relatório de Tickets (função por tenant)
# ==========================================
def extrair_tickets(t, p):
    """Baixa o relatório deste tenant. A empresa já foi selecionada pela Célula 7 (troca feita uma única vez por tenant)."""
    print(f"\n==========================================")
    print(f"📊 Processando Tickets: {t['nome_exibicao']} ({p['de_str']} até {p['ate_str']})")
    print(f"==========================================")
    
    try:
        
        driver.get(URL_TICKETS)
        
        input_from = wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtFromDate']")))
        input_to = driver.find_element(By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtToDate']")
        driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_from, p["de_str"])
        driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_to, p["ate_str"])
        driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_from)
        driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_to)
        
        btn_gerar = wait_curto.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[ga-event='mktzap_bt_action_history_tickets_report_generate']")))
        driver.execute_script("arguments[0].click();", btn_gerar)
        
        aguardar_e_mover_download(t["pasta_temp"], pasta_bruto(t, p), "bruto_tickets", p)
        
    except Exception as e:
        print(f"❌ Erro na extração dos Tickets ({t['nome_exibicao']}): {e}")
        registro_execucao.append({"tenant": t["nome_exibicao"], "periodo": p["pasta_mes"], "relatorio": "Tickets", "status": f"Erro: {str(e)}"})

In [ ]:
# ==========================================
# Célula 7 - Execução: por empresa, baixa Analítico, Enquetes e Tickets
# ==========================================
# Para cada tenant: entra na empresa UMA vez e baixa os 3 relatórios de cada período de PERIODOS (Célula 1),
# depois passa para a próxima.
for t in tenants:
    print("\n" + "=" * 70)
    print(f"🏢 {t['nome_exibicao']}  ->  empresa MKTZap: {t['nome_empresa_mktzap']}")
    print("=" * 70)

    try:
        definir_pasta_download(t["pasta_temp"])
        trocar_empresa(t["nome_empresa_mktzap"])
    except Exception as e:
        # Sem entrar na empresa certa não se baixa nada: evitaria gravar relatório de outra empresa neste tenant
        print(f"❌ Não foi possível entrar na empresa '{t['nome_empresa_mktzap']}': {e}")
        registro_execucao.append({"tenant": t["nome_exibicao"], "relatorio": "Troca de empresa", "status": f"Erro: {e}"})
        continue

    for p in PERIODOS:
        print(f"\n📅 Período: {p['de_str']} até {p['ate_str']}  ->  {pasta_bruto(t, p)}")
        extrair_analitico(t, p)
        extrair_enquetes(t, p)
        extrair_tickets(t, p)

print("\n✅ Execução concluída para todos os tenants. Rode a Célula 8 para encerrar o navegador e a 9 para conferir.")

In [ ]:
# ==========================================
# Célula 8 - Encerrar a Sessão do Selenium
# ==========================================
try:
    driver.quit()
    print("🔒 Sessão do Selenium finalizada com sucesso! Todos os arquivos brutos foram processados.")
except Exception as e:
    print(f"⚠️ Aviso ao encerrar o driver: {e}")
    fechar_chrome_do_pipeline()

In [ ]:
# ==========================================
# Célula 9 - Conferência dos downloads e Limpeza dos Temporários
# ==========================================
time.sleep(2)

# Quais relatórios chegaram (um por período, tenant e relatório) e se parecem mesmo um relatório
resumo = []
for p in PERIODOS:
    for t in tenants:
        for rel in ("analitico", "enquetes", "tickets"):
            arqs = glob.glob(os.path.join(pasta_bruto(t, p), f"bruto_{rel}_{p['ini']:%Y%m%d}_a_{p['fim']:%Y%m%d}.*"))
            if not arqs:
                resumo.append((p["pasta_mes"], t["nome_exibicao"], rel, "❌ NÃO BAIXADO", ""))
                continue
            arq = max(arqs, key=os.path.getmtime)
            with open(arq, "rb") as f:
                inicio = f.read(4)
            kb = os.path.getsize(arq) / 1024
            situacao = "⚠️ parece pacote do Chrome (.crx), não é relatório" if inicio == b"Cr24" else "✅ ok"
            resumo.append((p["pasta_mes"], t["nome_exibicao"], rel, situacao, f"{os.path.basename(arq)} ({kb:,.1f} KB)"))
for linha in resumo:
    print(f"{linha[0]:7} {linha[1]:16} {linha[2]:10} {linha[3]:12} {linha[4]}")
if registro_execucao:
    print("\n⚠️ FALHAS REGISTRADAS NA EXTRAÇÃO:")
    for r in registro_execucao:
        print("  ", r)

# Limpeza das pastas temporárias de download
def forcar_exclusao(func, path, exc_info):
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"⚠️ Aviso ao excluir {path}: {e}")

for t in tenants:
    if os.path.exists(t["pasta_temp"]):
        try:
            shutil.rmtree(t["pasta_temp"], onerror=forcar_exclusao)
        except Exception as e:
            print(f"⚠️ Pasta trancada no SO/OneDrive: {e}")
print("✨ Extração finalizada. Os brutos estão em:", PASTA_SAIDA)

In [ ]:
# ==========================================
# Célula 10 - Bruto Analítico Brasil -> estrutura do consolidado -> arquivo Gold
# ==========================================
# Para cada período de PERIODOS (Célula 1): 1) lê o bruto que o Selenium baixou para aquele período; 2) deixa as colunas
# iguais às do consolidado (mesmos nomes e ordem); 3) aplica a regra da 'Qtd Atendimentos'; 4) grava como
# <data>-brasil-analitico-periodo-<de>-a-<ate>.csv em pre-processing\<anomês do período>. O arquivo bruto original NÃO é alterado.
# Também define as funções comuns usadas pelas células seguintes (bruto do período, gravação do Gold, meses a substituir).
import csv
import html
import re
import pandas as pd
import openpyxl

CONSOLIDADO_BRASIL = os.path.join(RAIZ_FONTES, "EXP BR", "MKT", "MKT_Reativos.xlsx")

# Limpeza de texto dos VALORES das células (os nomes das colunas não são tocados), feita logo após ler o bruto:
#   - decodifica entidades HTML: N&#195;O -> NÃO, M&#170; -> Mª, &#127809; -> emoji (repete até estabilizar, p.ex. &amp;#195;)
#   - troca quebras de linha dentro da célula por espaço e tira espaços das pontas
# Coloque False para manter o texto exatamente como veio do MKTZap.
LIMPAR_TEXTO = True


def colunas_do_consolidado(caminho):
    wb = openpyxl.load_workbook(caminho, read_only=True)   # só leitura
    cab = list(next(wb.worksheets[0].iter_rows(max_row=1, values_only=True)))
    wb.close()
    while cab and cab[-1] is None:
        cab.pop()
    return [str(c) for c in cab]


def achar_cabecalho_bruto(caminho):
    """Linha do cabeçalho real (o export pode ter títulos antes), codificação e separador."""
    for enc in ("utf-8-sig", "latin1", "utf-16"):
        try:
            with open(caminho, "r", encoding=enc, newline="") as f:
                for idx, linha in enumerate(f):
                    m = linha.lower()
                    if "protocolo" in m or "protocol" in m or "data contato" in m:
                        sep = max((";", ",", "\t"), key=lambda s: len(next(csv.reader([linha], delimiter=s))))
                        return idx, enc, sep
        except UnicodeError:
            continue
    raise ValueError("Não encontrei a linha de cabeçalho do bruto (nenhuma linha com 'Protocolo' ou 'Data Contato').")


def regra_qtd_atendimentos(valor):
    """1 quando vazio, não numérico, <= 0 ou >= 100; senão o próprio número inteiro."""
    texto = str(valor).strip()
    if texto == "" or texto.lower() in ("none", "nan", "null"):
        return "1"
    try:
        n = float(texto.replace(",", "."))
    except ValueError:
        return "1"
    if n <= 0 or n >= 100:
        return "1"
    return str(int(n))


def limpar_texto_valor(v):
    """Decodifica entidades HTML (até estabilizar), troca quebras de linha por espaço e tira espaços das pontas."""
    for _ in range(3):                       # repete até não mudar mais (entidade dentro de entidade)
        novo = html.unescape(v) if "&" in v and ";" in v else v
        if novo == v:
            break
        v = novo
    return re.sub(r"\r\n|\r|\n", " ", v).strip()


def bruto_do_periodo(t, p, relatorio):
    """O bruto baixado para este período, pelo nome exato. Um bruto de outra execução não serve: gravado como mês
    inteiro, faria o mês parecer fechado sem os últimos dias."""
    arquivo = os.path.join(pasta_bruto(t, p), f"bruto_{relatorio}_{p['ini']:%Y%m%d}_a_{p['fim']:%Y%m%d}.csv")
    if not os.path.exists(arquivo):
        raise FileNotFoundError(f"Não encontrei o bruto deste período: {arquivo}. Rode a Célula 7 antes (e confira a Célula 9).")
    if os.path.getsize(arquivo) == 0:
        raise ValueError(f"O bruto está vazio (0 bytes), o download falhou: {arquivo}. Baixe de novo antes de consolidar.")
    return arquivo


def salvar_gold(df, t, p, slug, relatorio):
    """Grava em pre-processing/<anomês do período>/<tenant> como <data da execução>-<slug>-<relatorio>-periodo-<de>-a-<ate>.csv.
    Se já existir um arquivo com esse nome, guarda a versão anterior ao lado."""
    pasta = os.path.join(RAIZ_AUTO, "pre-processing", p["pasta_mes"], t["nome_exibicao"])
    os.makedirs(pasta, exist_ok=True)
    destino = os.path.join(pasta, f"{datetime.now():%Y-%m-%d}-{slug}-{relatorio}-periodo-{p['de_iso']}-a-{p['ate_iso']}.csv")
    if os.path.exists(destino):
        guardado = destino + f".anterior_{datetime.now():%H%M%S}"
        os.replace(destino, guardado)
        print(f"  -> Já existia um arquivo com esse nome; a versão anterior foi guardada como: {os.path.basename(guardado)}")
    df.to_csv(destino, sep=";", index=False, encoding="utf-8-sig")
    print(f"✅ Gravado: {destino}\n   {len(df):,} linhas x {len(df.columns)} colunas (mesma estrutura do consolidado)")
    return destino


def linhas_dos_periodos(datas, golds):
    """True nas linhas cujas datas caem em um dos meses de `golds` (lista de (período, arquivo Gold))."""
    mascara = pd.Series(False, index=datas.index)
    for p, _ in golds:
        ini = pd.Timestamp(p["ini"])
        mascara |= (datas >= ini) & (datas < ini + pd.offsets.MonthBegin(1))
    return mascara


tenant_brasil = next(t for t in tenants if t["id"] == "27")


def gold_analitico_brasil(p):
    # --- 1) bruto do Brasil neste período
    arquivo_bruto = bruto_do_periodo(tenant_brasil, p, "analitico")
    linha_cab, codificacao, separador = achar_cabecalho_bruto(arquivo_bruto)
    df = pd.read_csv(arquivo_bruto, skiprows=linha_cab, sep=separador, encoding=codificacao,
                     dtype=str, keep_default_na=False)   # sem pular linhas ruins: se houver problema, dá erro em vez de perder dados
    df.columns = df.columns.astype(str).str.strip()
    print(f"\n📅 {p['de_str']} até {p['ate_str']}\nBruto: {arquivo_bruto}\n  -> {len(df):,} linhas x {len(df.columns)} colunas")

    # --- 1b) limpeza de texto nos valores (antes de qualquer mexida nas colunas)
    if LIMPAR_TEXTO:
        antes_txt = df.copy()
        df = df.apply(lambda coluna: coluna.map(limpar_texto_valor))
        print(f"  -> Limpeza de texto: {int((antes_txt != df).to_numpy().sum()):,} células alteradas (nomes das colunas intactos)")
    else:
        print("  -> Limpeza de texto desligada (LIMPAR_TEXTO = False): o texto do bruto foi mantido como veio")

    # --- 2) mesma estrutura do consolidado
    colunas = colunas_do_consolidado(CONSOLIDADO_BRASIL)
    fora = [c for c in df.columns if c not in colunas]
    sem = [c for c in colunas if c not in df.columns]
    if fora:
        print(f"  ⚠️ Colunas do bruto que NÃO existem no consolidado e serão descartadas: {fora}")
    if sem:
        print(f"  -> {len(sem)} colunas do consolidado ausentes no bruto, criadas vazias: {sem}")
    df = df.reindex(columns=colunas, fill_value="")

    # --- 3) regra da Qtd Atendimentos
    antes = df["Qtd Atendimentos"].copy()
    df["Qtd Atendimentos"] = antes.map(regra_qtd_atendimentos)
    alteradas = int((antes.astype(str).str.strip() != df["Qtd Atendimentos"]).sum())
    print(f"  -> Qtd Atendimentos: {alteradas:,} valores trocados por 1 (vazio, não numérico, <= 0 ou >= 100)")

    # --- 4) grava com o nome final
    return salvar_gold(df, tenant_brasil, p, "brasil", "analitico")


golds_brasil = [(p, gold_analitico_brasil(p)) for p in PERIODOS]

In [ ]:
# ==========================================
# Célula 11 - Atualiza o consolidado Brasil: remove o(s) mês(es) dos períodos e cola os arquivos da Célula 10
# ==========================================
# Lê o MKT_Reativos.xlsx, remove as linhas cujo 'Data Contato' está em algum mês de PERIODOS (o atual e, na virada, também o
# anterior), cola todas as linhas dos arquivos gerados na Célula 10 e salva por cima do MKT_Reativos.xlsx (mesmo nome). Meses anteriores não são retipados. Depende da Célula 10.
DESTINO_CONSOLIDADO = CONSOLIDADO_BRASIL
GOLDS_NOVOS = golds_brasil             # (período, arquivo) gerados na Célula 10
COL_DATA = "Data Contato"
COLS_DATA = ["Data Contato", "Encerrado em"]
COLS_INTEIRAS = ["Qtd Atendimentos", "Qtde Mensagens", "Quantidade de Tickets"]
COLS_TEMPO = ["Tempo do atendimento", "Tempo de Operação", "Tempo até ser atendido", "Primeiro Tempo de Espera", "Tempo Médio de Espera"]
FORMATO_DATA_PADRAO = "DD/MM/YYYY HH:MM"
FORMATO_TEMPO = "[h]:mm:ss"


def para_data(v):
    """datetime nativo, ou texto dd/mm/aaaa [hh:mm[:ss]]; None se não for data."""
    if isinstance(v, datetime):
        return v
    if isinstance(v, str):
        for fmt in ("%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S"):
            try:
                return datetime.strptime(v.strip(), fmt)
            except ValueError:
                pass
    return None


def para_inteiro_excel(v):
    """Como o Excel ao colar: texto inteiro (sem zero à esquerda, até 15 dígitos; aceita '15.0') vira número."""
    if isinstance(v, str) and re.fullmatch(r"(?:0|[1-9]\d{0,14})(?:[.,]0+)?", v):
        return int(re.split(r"[.,]", v)[0])
    return v


def para_tempo(v):
    """'hh:mm:ss' vira hora de verdade (fração do dia), como o Excel ao colar; senão fica como está."""
    m = re.fullmatch(r"\s*(\d{1,3}):(\d{2}):(\d{2})\s*", v)
    return (int(m[1]) * 3600 + int(m[2]) * 60 + int(m[3])) / 86400 if m else (v if v != "" else None)


def ler_consolidado(caminho):
    """Lê a 1ª aba: valores como estão, cabeçalho exato e formatos de exibição das colunas."""
    wb = openpyxl.load_workbook(caminho, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    aba = ws.title
    linhas = ws.iter_rows(values_only=True)
    cab = list(next(linhas))
    while cab and cab[-1] is None:
        cab.pop()
    n = len(cab)
    dados = [list(r[:n]) + [None] * (n - len(r[:n])) for r in linhas if any(v is not None for v in r[:n])]
    wb.close()
    wb = openpyxl.load_workbook(caminho, read_only=True)
    formatos = {}
    for row in wb.worksheets[0].iter_rows(min_row=2, max_row=501, max_col=n):
        for i, c in enumerate(row):
            if i not in formatos and isinstance(c.value, (datetime, int, float)) and not isinstance(c.value, bool):
                formatos[i] = c.number_format
    wb.close()
    return pd.DataFrame(dados, columns=[str(c) for c in cab], dtype=object), aba, cab, formatos


def preparar_novas_linhas(caminho_csv):
    """Lê o arquivo da Célula 10 e converte como o Excel ao colar: datas e inteiros nativos, vazio = célula vazia."""
    novo = pd.read_csv(caminho_csv, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)
    for c in novo.columns:
        if c in COLS_DATA:
            novo[c] = novo[c].map(lambda v: para_data(v) if v.strip() else None).astype(object)
        elif c in COLS_TEMPO:
            novo[c] = novo[c].map(para_tempo).astype(object)
        elif c in COLS_INTEIRAS:
            novo[c] = novo[c].map(lambda v: int(float(v.replace(",", "."))) if re.fullmatch(r"\s*-?\d+([.,]\d+)?\s*", v) else None).astype(object)
        else:
            novo[c] = novo[c].map(lambda v: para_inteiro_excel(v) if v != "" else None).astype(object)
    return novo


def gravar_no_lugar(df, caminho, aba, cabecalho, formatos):
    """Troca só a aba (o arquivo e o File ID do Drive permanecem), com cabeçalho e formatos de exibição originais."""
    formula = []
    for j, c in enumerate(df.columns):
        formula += [(i, j) for i in df.index[df[c].map(lambda v: isinstance(v, str) and v.startswith("="))]]
    with pd.ExcelWriter(caminho, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
        df.to_excel(w, sheet_name=aba, index=False)
        ws = w.sheets[aba]
        for j, nome in enumerate(cabecalho, start=1):
            ws.cell(row=1, column=j).value = nome          # grafia original do cabeçalho
        for i, j in formula:                               # textos iniciados por '=' ficam como texto, não fórmula
            ws.cell(row=i + 2, column=j + 1).value = df.iat[i, j]
            ws.cell(row=i + 2, column=j + 1).data_type = "s"
        for idx, c in enumerate(df.columns, start=1):
            if c in COLS_DATA:
                fmt, tipo = formatos.get(idx - 1, FORMATO_DATA_PADRAO), datetime
            elif c in COLS_INTEIRAS:
                fmt, tipo = formatos.get(idx - 1, "0"), int
            elif c in COLS_TEMPO:
                fmt, tipo = FORMATO_TEMPO, float
            else:
                continue
            for (cel,) in ws.iter_rows(min_row=2, max_row=len(df) + 1, min_col=idx, max_col=idx):
                if isinstance(cel.value, tipo):
                    cel.number_format = fmt


historico, aba, cabecalho, formatos = ler_consolidado(DESTINO_CONSOLIDADO)
novas = pd.concat([preparar_novas_linhas(g) for _, g in GOLDS_NOVOS], ignore_index=True)

d_hist = pd.to_datetime(historico[COL_DATA].map(para_data), errors="coerce")
no_mes = linhas_dos_periodos(d_hist, GOLDS_NOVOS)          # linhas dos meses que estão sendo substituídos
final = pd.concat([historico[~no_mes], novas], ignore_index=True)

gravar_no_lugar(final, DESTINO_CONSOLIDADO, aba, cabecalho, formatos)
meses = ", ".join(f"{p['ini']:%m/%Y}" for p, _ in GOLDS_NOVOS)
print(f"✅ {os.path.basename(DESTINO_CONSOLIDADO)} atualizado: {int(no_mes.sum()):,} linhas de {meses} removidas, "
      f"{len(novas):,} coladas, {len(final):,} no total.")

In [ ]:
# ==========================================
# Célula 12 - Bruto Analítico Hispana -> estrutura do consolidado -> arquivo Gold
# ==========================================
# Para cada período de PERIODOS: 1) lê o bruto que o Selenium baixou; 2) limpa os textos acentuados; 3) renomeia
# 'PEDIDOS FINALIZADOS' para 'Pedidos Finalizados'; 4) grava como <data>-hispana-analitico-periodo-<de>-a-<ate>.csv em
# pre-processing. O bruto original NÃO é alterado. Depende da Célula 1 e da Célula 10 (funções e LIMPAR_TEXTO).
CONSOLIDADO_HISPANA = os.path.join(RAIZ_FONTES, "EXP OIS", "MKT", "MKT Analitico Hispana.xlsx")
RENOMEAR_COLUNAS_HISPANA = {"PEDIDOS FINALIZADOS": "Pedidos Finalizados"}
tenant_hispana = next(t for t in tenants if t["id"] == "1531")


def gold_analitico_hispana(p):
    # --- 1) bruto da Hispana neste período
    arquivo_bruto_h = bruto_do_periodo(tenant_hispana, p, "analitico")
    linha_cab_h, codificacao_h, separador_h = achar_cabecalho_bruto(arquivo_bruto_h)
    df_h = pd.read_csv(arquivo_bruto_h, skiprows=linha_cab_h, sep=separador_h, encoding=codificacao_h,
                       dtype=str, keep_default_na=False)
    df_h.columns = df_h.columns.astype(str).str.strip()
    print(f"\n📅 {p['de_str']} até {p['ate_str']}\nBruto: {arquivo_bruto_h}\n  -> {len(df_h):,} linhas x {len(df_h.columns)} colunas")

    # --- 2) limpeza de texto nos valores (antes de mexer nos nomes das colunas)
    if LIMPAR_TEXTO:
        antes_h = df_h.copy()
        df_h = df_h.apply(lambda coluna: coluna.map(limpar_texto_valor))
        print(f"  -> Limpeza de texto: {int((antes_h != df_h).to_numpy().sum()):,} células alteradas (nomes das colunas intactos)")

    # --- 3) nome da coluna igual ao do consolidado
    df_h = df_h.rename(columns=RENOMEAR_COLUNAS_HISPANA)
    print(f"  -> Colunas renomeadas: {RENOMEAR_COLUNAS_HISPANA}")

    # --- 4) grava com o nome final
    return salvar_gold(df_h, tenant_hispana, p, "hispana", "analitico")


golds_hispana = [(p, gold_analitico_hispana(p)) for p in PERIODOS]

In [ ]:
# ==========================================
# Célula 13 - Atualiza o consolidado Hispana: remove o(s) mês(es) dos períodos e cola os arquivos da Célula 12
# ==========================================
# Lê o 'MKT Analitico Hispana.xlsx', remove as linhas cujo 'Data Contato' está em algum mês de PERIODOS, cola todas as linhas dos
# arquivos gerados na Célula 12 e salva por cima do mesmo arquivo. Meses anteriores não são retipados. As colunas de tempo ficam como texto
# (assim estão no consolidado da Hispana). Independente da Célula 11; depende das Células 1 e 12.
DESTINO_CONSOLIDADO_H = CONSOLIDADO_HISPANA
GOLDS_NOVOS_H = golds_hispana          # (período, arquivo) gerados na Célula 12
COL_DATA_H = "Data Contato"
COLS_DATA_H = ["Data Contato", "Encerrado em"]
COLS_INTEIRAS_H = ["Qtde Mensagens", "Quantidade de Tickets"]
FORMATO_DATA_PADRAO_H = "DD/MM/YYYY HH:MM:SS"


def para_data_h(v):
    """datetime nativo, ou texto dd/mm/aaaa [hh:mm[:ss]]; None se não for data."""
    if isinstance(v, datetime):
        return v
    if isinstance(v, str):
        for fmt in ("%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S"):
            try:
                return datetime.strptime(v.strip(), fmt)
            except ValueError:
                pass
    return None


def para_inteiro_h(v):
    """Como o Excel ao colar: texto inteiro (sem zero à esquerda, até 15 dígitos; aceita '15.0') vira número."""
    if isinstance(v, str) and re.fullmatch(r"(?:0|[1-9]\d{0,14})(?:[.,]0+)?", v):
        return int(re.split(r"[.,]", v)[0])
    return v


def ler_consolidado_h(caminho):
    """Lê a 1ª aba: valores como estão, cabeçalho exato e formatos de exibição das colunas."""
    wb = openpyxl.load_workbook(caminho, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    aba = ws.title
    linhas = ws.iter_rows(values_only=True)
    cab = list(next(linhas))
    while cab and cab[-1] is None:
        cab.pop()
    n = len(cab)
    dados = [list(r[:n]) + [None] * (n - len(r[:n])) for r in linhas if any(v is not None for v in r[:n])]
    wb.close()
    wb = openpyxl.load_workbook(caminho, read_only=True)
    formatos = {}
    for row in wb.worksheets[0].iter_rows(min_row=2, max_row=501, max_col=n):
        for i, c in enumerate(row):
            if i not in formatos and isinstance(c.value, (datetime, int, float)) and not isinstance(c.value, bool):
                formatos[i] = c.number_format
    wb.close()
    return pd.DataFrame(dados, columns=[str(c) for c in cab], dtype=object), aba, cab, formatos


def preparar_novas_linhas_h(caminho_csv):
    """Lê o arquivo da Célula 12 e converte como o Excel ao colar: datas e números nativos, vazio = célula vazia."""
    novo = pd.read_csv(caminho_csv, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)
    for c in novo.columns:
        if c in COLS_DATA_H:
            novo[c] = novo[c].map(lambda v: para_data_h(v) if v.strip() else None).astype(object)
        elif c in COLS_INTEIRAS_H:
            novo[c] = novo[c].map(lambda v: int(float(v.replace(",", "."))) if re.fullmatch(r"\s*-?\d+([.,]\d+)?\s*", v) else None).astype(object)
        else:
            novo[c] = novo[c].map(lambda v: para_inteiro_h(v) if v != "" else None).astype(object)
    return novo


def gravar_no_lugar_h(df, caminho, aba, cabecalho, formatos):
    """Troca só a aba (o arquivo e o File ID do Drive permanecem), com cabeçalho e formatos de exibição originais."""
    formula = []
    for j, c in enumerate(df.columns):
        formula += [(i, j) for i in df.index[df[c].map(lambda v: isinstance(v, str) and v.startswith("="))]]
    with pd.ExcelWriter(caminho, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
        df.to_excel(w, sheet_name=aba, index=False)
        ws = w.sheets[aba]
        for j, nome in enumerate(cabecalho, start=1):
            ws.cell(row=1, column=j).value = nome          # grafia original do cabeçalho
        for i, j in formula:                               # textos iniciados por '=' ficam como texto, não fórmula
            ws.cell(row=i + 2, column=j + 1).value = df.iat[i, j]
            ws.cell(row=i + 2, column=j + 1).data_type = "s"
        for idx, c in enumerate(df.columns, start=1):
            if c in COLS_DATA_H:
                fmt, tipo = formatos.get(idx - 1, FORMATO_DATA_PADRAO_H), datetime
            elif c in COLS_INTEIRAS_H:
                fmt, tipo = formatos.get(idx - 1, "0"), int
            else:
                continue
            for (cel,) in ws.iter_rows(min_row=2, max_row=len(df) + 1, min_col=idx, max_col=idx):
                if isinstance(cel.value, tipo):
                    cel.number_format = fmt


historico_h, aba_h, cabecalho_h, formatos_h = ler_consolidado_h(DESTINO_CONSOLIDADO_H)
novas_h = pd.concat([preparar_novas_linhas_h(g) for _, g in GOLDS_NOVOS_H], ignore_index=True)

d_hist_h = pd.to_datetime(historico_h[COL_DATA_H].map(para_data_h), errors="coerce")
no_mes_h = linhas_dos_periodos(d_hist_h, GOLDS_NOVOS_H)      # linhas dos meses que estão sendo substituídos
final_h = pd.concat([historico_h[~no_mes_h], novas_h], ignore_index=True)

gravar_no_lugar_h(final_h, DESTINO_CONSOLIDADO_H, aba_h, cabecalho_h, formatos_h)
meses_h = ", ".join(f"{p['ini']:%m/%Y}" for p, _ in GOLDS_NOVOS_H)
print(f"✅ {os.path.basename(DESTINO_CONSOLIDADO_H)} atualizado: {int(no_mes_h.sum()):,} linhas de {meses_h} removidas, "
      f"{len(novas_h):,} coladas, {len(final_h):,} no total.")

In [ ]:
# ==========================================
# Célula 14 - Bruto Analítico NCC -> estrutura do consolidado -> arquivo Gold
# ==========================================
# Para cada período de PERIODOS: 1) lê o bruto que o Selenium baixou; 2) limpa os textos acentuados; 3) monta as colunas na
# ordem do consolidado (BASE MALASIA.xlsx; as que só existem nele ficam vazias; 'telefone' é descartada); 4) grava como
# <data>-ncc-analitico-periodo-<de>-a-<ate>.csv em pre-processing. O bruto original NÃO é alterado.
# O bruto e o consolidado têm duas colunas 'Nome' (o pandas chama a 2ª do bruto de 'Nome.1'): cada uma vai para a sua posição.
# Depende da Célula 1 e da Célula 10 (funções e LIMPAR_TEXTO).
CONSOLIDADO_NCC = os.path.join(RAIZ_FONTES, "Enquetes", "BASE MALASIA.xlsx")
tenant_ncc = next(t for t in tenants if t["id"] == "1990")


def gold_analitico_ncc(p):
    # --- 1) bruto do NCC neste período
    arquivo_bruto_n = bruto_do_periodo(tenant_ncc, p, "analitico")
    linha_cab_n, codificacao_n, separador_n = achar_cabecalho_bruto(arquivo_bruto_n)
    df_n = pd.read_csv(arquivo_bruto_n, skiprows=linha_cab_n, sep=separador_n, encoding=codificacao_n,
                       dtype=str, keep_default_na=False)
    df_n.columns = df_n.columns.astype(str).str.strip()
    print(f"\n📅 {p['de_str']} até {p['ate_str']}\nBruto: {arquivo_bruto_n}\n  -> {len(df_n):,} linhas x {len(df_n.columns)} colunas")

    # --- 2) limpeza de texto nos valores (antes de mexer nos nomes das colunas)
    if LIMPAR_TEXTO:
        antes_n = df_n.copy()
        df_n = df_n.apply(lambda coluna: coluna.map(limpar_texto_valor))
        print(f"  -> Limpeza de texto: {int((antes_n != df_n).to_numpy().sum()):,} células alteradas (nomes das colunas intactos)")

    # --- 3) mesma estrutura do consolidado (nomes repetidos: 1ª ocorrência = 'Nome', 2ª = 'Nome.1', como o pandas lê o bruto)
    colunas_n = colunas_do_consolidado(CONSOLIDADO_NCC)
    vistos, chaves_n = {}, []
    for c in colunas_n:
        k = vistos.get(c, 0)
        chaves_n.append(c if k == 0 else f"{c}.{k}")
        vistos[c] = k + 1
    descartadas = [c for c in df_n.columns if c not in chaves_n]
    vazias = [c for c in chaves_n if c not in df_n.columns]
    print(f"  -> Colunas do bruto descartadas (não existem no consolidado): {descartadas}")
    print(f"  -> {len(vazias)} colunas do consolidado ausentes no bruto, criadas vazias: {vazias}")
    df_n = df_n.reindex(columns=chaves_n, fill_value="")
    df_n.columns = colunas_n

    # --- 4) grava com o nome final
    return salvar_gold(df_n, tenant_ncc, p, "ncc", "analitico")


golds_ncc = [(p, gold_analitico_ncc(p)) for p in PERIODOS]

In [ ]:
# ==========================================
# Célula 15 - Atualiza o consolidado NCC: remove o(s) mês(es) dos períodos e cola os arquivos da Célula 14
# ==========================================
# Lê o 'BASE MALASIA.xlsx', remove as linhas cujo 'Data Contato' está em algum mês de PERIODOS, cola todas as linhas dos arquivos
# gerados na Célula 14 e salva por cima do mesmo arquivo. Meses anteriores não são retipados. As colunas de tempo ficam como texto
# (assim estão no consolidado do NCC). As duas colunas 'Nome' são tratadas como 'Nome' e 'Nome.1' dentro do código e voltam
# a se chamar 'Nome' no arquivo. Independente das Células 11 e 13; depende das Células 1 e 16.
DESTINO_CONSOLIDADO_N = CONSOLIDADO_NCC
GOLDS_NOVOS_N = golds_ncc              # (período, arquivo) gerados na Célula 14
COL_DATA_N = "Data Contato"
COLS_DATA_N = ["Data Contato", "Encerrado em"]
COLS_INTEIRAS_N = ["Qtde Mensagens", "Quantidade de Tickets"]
FORMATO_DATA_PADRAO_N = "DD/MM/YYYY HH:MM:SS"


def chaves_unicas_n(nomes):
    """Nomes repetidos viram 'Nome', 'Nome.1', ... (como o pandas lê o CSV)."""
    vistos, chaves = {}, []
    for c in nomes:
        k = vistos.get(c, 0)
        chaves.append(c if k == 0 else f"{c}.{k}")
        vistos[c] = k + 1
    return chaves


def para_data_n(v):
    """datetime nativo, ou texto dd/mm/aaaa [hh:mm[:ss]]; None se não for data."""
    if isinstance(v, datetime):
        return v
    if isinstance(v, str):
        for fmt in ("%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S"):
            try:
                return datetime.strptime(v.strip(), fmt)
            except ValueError:
                pass
    return None


def para_inteiro_n(v):
    """Como o Excel ao colar: texto inteiro (sem zero à esquerda, até 15 dígitos; aceita '15.0') vira número."""
    if isinstance(v, str) and re.fullmatch(r"(?:0|[1-9]\d{0,14})(?:[.,]0+)?", v):
        return int(re.split(r"[.,]", v)[0])
    return v


def ler_consolidado_n(caminho):
    """Lê a 1ª aba: valores como estão, cabeçalho exato e formatos de exibição das colunas."""
    wb = openpyxl.load_workbook(caminho, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    aba = ws.title
    linhas = ws.iter_rows(values_only=True)
    cab = list(next(linhas))
    while cab and cab[-1] is None:
        cab.pop()
    n = len(cab)
    dados = [list(r[:n]) + [None] * (n - len(r[:n])) for r in linhas if any(v is not None for v in r[:n])]
    wb.close()
    wb = openpyxl.load_workbook(caminho, read_only=True)
    formatos = {}
    for row in wb.worksheets[0].iter_rows(min_row=2, max_row=501, max_col=n):
        for i, c in enumerate(row):
            if i not in formatos and isinstance(c.value, (datetime, int, float)) and not isinstance(c.value, bool):
                formatos[i] = c.number_format
    wb.close()
    return pd.DataFrame(dados, columns=chaves_unicas_n([str(c) for c in cab]), dtype=object), aba, cab, formatos


def preparar_novas_linhas_n(caminho_csv):
    """Lê o arquivo da Célula 16 e converte como o Excel ao colar: datas e números nativos, vazio = célula vazia."""
    novo = pd.read_csv(caminho_csv, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)   # 2ª 'Nome' vira 'Nome.1'
    for c in novo.columns:
        if c in COLS_DATA_N:
            novo[c] = novo[c].map(lambda v: para_data_n(v) if v.strip() else None).astype(object)
        elif c in COLS_INTEIRAS_N:
            novo[c] = novo[c].map(lambda v: int(float(v.replace(",", "."))) if re.fullmatch(r"\s*-?\d+([.,]\d+)?\s*", v) else None).astype(object)
        else:
            novo[c] = novo[c].map(lambda v: para_inteiro_n(v) if v != "" else None).astype(object)
    return novo


def gravar_no_lugar_n(df, caminho, aba, cabecalho, formatos):
    """Troca só a aba (o arquivo e o File ID do Drive permanecem), com cabeçalho e formatos de exibição originais."""
    formula = []
    for j, c in enumerate(df.columns):
        formula += [(i, j) for i in df.index[df[c].map(lambda v: isinstance(v, str) and v.startswith("="))]]
    with pd.ExcelWriter(caminho, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
        df.to_excel(w, sheet_name=aba, index=False)
        ws = w.sheets[aba]
        for j, nome in enumerate(cabecalho, start=1):
            ws.cell(row=1, column=j).value = nome          # grafia original do cabeçalho (inclui as duas 'Nome')
        for i, j in formula:                               # textos iniciados por '=' ficam como texto, não fórmula
            ws.cell(row=i + 2, column=j + 1).value = df.iat[i, j]
            ws.cell(row=i + 2, column=j + 1).data_type = "s"
        for idx, c in enumerate(df.columns, start=1):
            if c in COLS_DATA_N:
                fmt, tipo = formatos.get(idx - 1, FORMATO_DATA_PADRAO_N), datetime
            elif c in COLS_INTEIRAS_N:
                fmt, tipo = formatos.get(idx - 1, "0"), int
            else:
                continue
            for (cel,) in ws.iter_rows(min_row=2, max_row=len(df) + 1, min_col=idx, max_col=idx):
                if isinstance(cel.value, tipo):
                    cel.number_format = fmt


historico_n, aba_n, cabecalho_n, formatos_n = ler_consolidado_n(DESTINO_CONSOLIDADO_N)
novas_n = pd.concat([preparar_novas_linhas_n(g) for _, g in GOLDS_NOVOS_N], ignore_index=True)

d_hist_n = pd.to_datetime(historico_n[COL_DATA_N].map(para_data_n), errors="coerce")
no_mes_n = linhas_dos_periodos(d_hist_n, GOLDS_NOVOS_N)      # linhas dos meses que estão sendo substituídos
final_n = pd.concat([historico_n[~no_mes_n], novas_n], ignore_index=True)

gravar_no_lugar_n(final_n, DESTINO_CONSOLIDADO_N, aba_n, cabecalho_n, formatos_n)
meses_n = ", ".join(f"{p['ini']:%m/%Y}" for p, _ in GOLDS_NOVOS_N)
print(f"✅ {os.path.basename(DESTINO_CONSOLIDADO_N)} atualizado: {int(no_mes_n.sum()):,} linhas de {meses_n} removidas, "
      f"{len(novas_n):,} coladas, {len(final_n):,} no total.")

In [ ]:
# ==========================================
# Célula 16 - Funções de tratamento e consolidação (Enquetes e Tickets)
# ==========================================
# Mesma lógica das bases Analítico, em funções reaproveitadas pelas células seguintes (uma célula de Gold e uma de
# consolidação por base). Depende das Células 1 e 10 (achar_cabecalho_bruto, colunas_do_consolidado, limpar_texto_valor,
# bruto_do_periodo, salvar_gold, linhas_dos_periodos, LIMPAR_TEXTO).


def gerar_gold(p, id_tenant, relatorio, consolidado, slug, renomear=None):
    """Bruto do período `p` -> limpeza de texto -> colunas na ordem do consolidado -> CSV Gold em pre-processing."""
    t = next(x for x in tenants if x["id"] == id_tenant)
    arquivo = bruto_do_periodo(t, p, relatorio)
    linha_cab, codificacao, separador = achar_cabecalho_bruto(arquivo)
    df = pd.read_csv(arquivo, skiprows=linha_cab, sep=separador, encoding=codificacao, dtype=str, keep_default_na=False)
    df.columns = df.columns.astype(str).str.strip()
    print(f"\n📅 {p['de_str']} até {p['ate_str']}\nBruto: {arquivo}\n  -> {len(df):,} linhas x {len(df.columns)} colunas")

    if LIMPAR_TEXTO:
        antes = df.copy()
        df = df.apply(lambda coluna: coluna.map(limpar_texto_valor))
        print(f"  -> Limpeza de texto: {int((antes != df).to_numpy().sum()):,} células alteradas (nomes das colunas intactos)")

    if renomear:
        df = df.rename(columns=renomear)
        print(f"  -> Colunas renomeadas: {renomear}")
    colunas = colunas_do_consolidado(consolidado)
    descartadas = [c for c in df.columns if c not in colunas]
    vazias = [c for c in colunas if c not in df.columns]
    if descartadas:
        print(f"  ⚠️ Colunas do bruto descartadas (não existem no consolidado): {descartadas}")
    if vazias:
        print(f"  -> Colunas do consolidado ausentes no bruto, criadas vazias: {vazias}")
    df = df.reindex(columns=colunas, fill_value="")
    return salvar_gold(df, t, p, slug, relatorio)


def gerar_golds(id_tenant, relatorio, consolidado, slug, renomear=None):
    """Um Gold por período de PERIODOS; devolve a lista de (período, arquivo) usada por consolidar()."""
    return [(p, gerar_gold(p, id_tenant, relatorio, consolidado, slug, renomear)) for p in PERIODOS]


def _para_data(v):
    if isinstance(v, datetime):
        return v
    if isinstance(v, str):
        for fmt in ("%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
            try:
                return datetime.strptime(v.strip(), fmt)
            except ValueError:
                pass
    return None


def consolidar(consolidado, golds, col_mes, cols_data, cols_numero, formato_data="DD/MM/YYYY HH:MM:SS"):
    """Remove do consolidado as linhas cujo `col_mes` está em algum mês de `golds` (lista de (período, arquivo) de gerar_golds),
    cola as linhas desses Golds e salva por cima (mesmo arquivo).
    Linhas novas: `cols_data` viram data e hora, `cols_numero` viram número (se forem números), o resto fica texto,
    como já está no consolidado. Meses anteriores não são retipados."""
    wb = openpyxl.load_workbook(consolidado, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    aba = ws.title
    linhas = ws.iter_rows(values_only=True)
    cab = list(next(linhas))
    while cab and cab[-1] is None:
        cab.pop()
    n = len(cab)
    dados = [list(r[:n]) + [None] * (n - len(r[:n])) for r in linhas if any(v is not None for v in r[:n])]
    wb.close()
    wb = openpyxl.load_workbook(consolidado, read_only=True)
    formatos = {}
    for row in wb.worksheets[0].iter_rows(min_row=2, max_row=501, max_col=n):
        for i, c in enumerate(row):
            if i not in formatos and isinstance(c.value, (datetime, int, float)) and not isinstance(c.value, bool):
                formatos[i] = c.number_format
    wb.close()
    historico = pd.DataFrame(dados, columns=[str(c) for c in cab], dtype=object)

    novas = pd.concat([pd.read_csv(g, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False) for _, g in golds],
                      ignore_index=True)
    novas.columns = historico.columns
    for c in novas.columns:
        if c in cols_data:
            novas[c] = novas[c].map(lambda v: (_para_data(v) or v) if v.strip() else None).astype(object)   # 'Não respondido' fica texto
        elif c in cols_numero:
            novas[c] = novas[c].map(lambda v: (int(float(v.replace(",", "."))) if re.fullmatch(r"\s*-?\d+([.,]0+)?\s*", v)
                                               else float(v.replace(",", ".")) if re.fullmatch(r"\s*-?\d+[.,]\d+\s*", v)
                                               else (v if v != "" else None))).astype(object)
        else:
            novas[c] = novas[c].map(lambda v: v if v != "" else None).astype(object)

    d_hist = pd.to_datetime(historico[col_mes].map(_para_data), errors="coerce")
    no_mes = linhas_dos_periodos(d_hist, golds)
    final = pd.concat([historico[~no_mes], novas], ignore_index=True)

    formula = []
    for j, c in enumerate(final.columns):
        formula += [(i, j) for i in final.index[final[c].map(lambda v: isinstance(v, str) and v.startswith("="))]]
    with pd.ExcelWriter(consolidado, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
        final.to_excel(w, sheet_name=aba, index=False)
        ws = w.sheets[aba]
        for j, nome in enumerate(cab, start=1):
            ws.cell(row=1, column=j).value = nome              # grafia original do cabeçalho
        for i, j in formula:                                   # textos iniciados por '=' ficam como texto, não fórmula
            ws.cell(row=i + 2, column=j + 1).value = final.iat[i, j]
            ws.cell(row=i + 2, column=j + 1).data_type = "s"
        for idx, c in enumerate(final.columns, start=1):
            if c in cols_data:
                fmt, tipo = formatos.get(idx - 1, formato_data), datetime
            elif c in cols_numero:
                fmt, tipo = formatos.get(idx - 1, "General"), (int, float)
            else:
                continue
            for (cel,) in ws.iter_rows(min_row=2, max_row=len(final) + 1, min_col=idx, max_col=idx):
                if isinstance(cel.value, tipo) and not isinstance(cel.value, bool):
                    cel.number_format = fmt
    meses = ", ".join(f"{p['ini']:%m/%Y}" for p, _ in golds)
    print(f"✅ {os.path.basename(consolidado)} atualizado: {int(no_mes.sum()):,} linhas de {meses} removidas (por '{col_mes}'), "
          f"{len(novas):,} coladas, {len(final):,} no total.")


print("✅ Funções de Enquetes e Tickets carregadas.")

In [ ]:
# ==========================================
# Célula 17 - Enquetes Brasil: bruto -> estrutura do consolidado -> arquivo Gold
# ==========================================
# No consolidado o cabeçalho 'Responsável' está gravado como 'ResponsÃ¡vel' (assim fica, para não mudar a base).
CONSOLIDADO_ENQ_BR = os.path.join(RAIZ_FONTES, "Enquetes", "Enquete MKT REATIVOS.xlsx")
golds_enq_br = gerar_golds("27", "enquetes", CONSOLIDADO_ENQ_BR, "brasil", renomear={"Responsável": "ResponsÃ¡vel"})

In [ ]:
# ==========================================
# Célula 18 - Enquetes Brasil: remove o(s) mês(es) dos períodos e cola os arquivos da Célula 19
# ==========================================
# O MKTZap filtra a exportação de Enquetes pela data de ENVIO, então o mês é separado por 'Enviado em'.
# 'Respondido em' vira data quando é data; 'Não respondido' fica texto. 'Nota' fica texto, como no consolidado.
consolidar(CONSOLIDADO_ENQ_BR, golds_enq_br,
           col_mes="Enviado em",
           cols_data=["Data do Contato", "Enviado em", "Respondido em"],
           cols_numero=[])

In [ ]:
# ==========================================
# Célula 19 - Enquetes Hispana: bruto -> estrutura do consolidado -> arquivo Gold
# ==========================================
# No consolidado o cabeçalho 'Responsável' está gravado como 'ResponsÃ¡vel' (assim fica, para não mudar a base).
CONSOLIDADO_ENQ_HI = os.path.join(RAIZ_FONTES, "Enquetes", "Enquete BASE MTK.xlsx")
golds_enq_hi = gerar_golds("1531", "enquetes", CONSOLIDADO_ENQ_HI, "hispana", renomear={"Responsável": "ResponsÃ¡vel"})

In [ ]:
# ==========================================
# Célula 20 - Enquetes Hispana: remove o(s) mês(es) dos períodos e cola os arquivos da Célula 21
# ==========================================
# O MKTZap filtra a exportação de Enquetes pela data de ENVIO, então o mês é separado por 'Enviado em'.
# 'Respondido em' vira data quando é data; 'Não respondido' fica texto. 'Nota' fica texto, como no consolidado.
consolidar(CONSOLIDADO_ENQ_HI, golds_enq_hi,
           col_mes="Enviado em",
           cols_data=["Data do Contato", "Enviado em", "Respondido em"],
           cols_numero=[])

In [ ]:
# ==========================================
# Célula 21 - Enquetes NCC: bruto -> estrutura do consolidado -> arquivo Gold
# ==========================================
# No consolidado o cabeçalho 'Responsável' está gravado como 'ResponsÃ¡vel' (assim fica, para não mudar a base).
CONSOLIDADO_ENQ_NCC = os.path.join(RAIZ_FONTES, "Enquetes", "Enquete Malasia.xlsx")
golds_enq_ncc = gerar_golds("1990", "enquetes", CONSOLIDADO_ENQ_NCC, "ncc", renomear={"Responsável": "ResponsÃ¡vel"})

In [ ]:
# ==========================================
# Célula 22 - Enquetes NCC: remove o(s) mês(es) dos períodos e cola os arquivos da Célula 23
# ==========================================
# O MKTZap filtra a exportação de Enquetes pela data de ENVIO, então o mês é separado por 'Enviado em'.
# 'Respondido em' vira data quando é data; 'Não respondido' fica texto. 'Nota' fica texto, como no consolidado.
consolidar(CONSOLIDADO_ENQ_NCC, golds_enq_ncc,
           col_mes="Enviado em",
           cols_data=["Data do Contato", "Enviado em", "Respondido em"],
           cols_numero=[])

In [ ]:
# ==========================================
# Célula 23 - Tickets Brasil: bruto -> estrutura do consolidado -> arquivo Gold
# ==========================================
CONSOLIDADO_TK_BR = os.path.join(RAIZ_FONTES, "EXP BR", "MKT", "Tickets Brasil.xlsx")
golds_tk_br = gerar_golds("27", "tickets", CONSOLIDADO_TK_BR, "brasil")

In [ ]:
# ==========================================
# Célula 24 - Tickets Brasil: remove o(s) mês(es) dos períodos e cola os arquivos da Célula 25
# ==========================================
# A exportação de Tickets é filtrada pela criação do ticket, então o mês é separado por 'Data de Criação do Ticket'.
consolidar(CONSOLIDADO_TK_BR, golds_tk_br,
           col_mes="Data de Criação do Ticket",
           cols_data=["Data de Criação do Atendimento", "Data de Finalização", "Data de Criação do Ticket", "Data de Conversão do Atendimento"],
           cols_numero=["Qtd Atendimentos", "Ciclo Vigente"])

In [ ]:
# ==========================================
# Célula 25 - Tickets Hispana: bruto -> estrutura do consolidado -> arquivo Gold
# ==========================================
CONSOLIDADO_TK_HI = os.path.join(RAIZ_FONTES, "EXP OIS", "MKT", "Tickets Hispana.xlsx")
golds_tk_hi = gerar_golds("1531", "tickets", CONSOLIDADO_TK_HI, "hispana")

In [ ]:
# ==========================================
# Célula 26 - Tickets Hispana: remove o(s) mês(es) dos períodos e cola os arquivos da Célula 27
# ==========================================
# A exportação de Tickets é filtrada pela criação do ticket, então o mês é separado por 'Data de Criação do Ticket'.
consolidar(CONSOLIDADO_TK_HI, golds_tk_hi,
           col_mes="Data de Criação do Ticket",
           cols_data=["Data de Criação do Atendimento", "Data de Finalização", "Data de Criação do Ticket", "Data de Conversão do Atendimento"],
           cols_numero=["Cant. de Atendimientos"])